v62 
- convert codebase for RL agent  
- headless scorer
- cross-sectional normalization
- discovery composite action
- temporal vectorizer

v61 
- **Verified all metric calculation with Excel** 
- Updated core.analyzer
- Replaced the `Result` pattern with exceptions and flattened the logic
- Refactored the `AlphaEngine` into a streamlined Orchestrator pattern
-  **Strict Date Logic:** No more "Time Travel" bugs.
-  **Dataclass Contracts:** No more "Magic String" typos or blind dictionaries.
-  **Exception Flow:** The `run` method is now a clean, readable story instead of a maze of `if/else` checks.
-  **Performance Workers:** Math is separated from orchestration.
- Ret_1d, explicitly turns division-by-zero results (`inf`) into `NaN`, and replace [np.inf, -np.inf] with np.nan



v60  
- Converted code from notebook to modular system.
- Fixed divide by zero warning from calculate_gain
- Added subtitle to subplots
- Added Volatility Regime plot


v59  
- Removed "nest" of if-statements in **AlphaEngine.run**
- Use **Result Pattern** to handle errors
- Change verify_analyzer_short and verify_analyzer_long gain calculation from simple return to logarithmic return
- Change calculate_gain from simple return to logarithmic return
- Remove bfill from calculate_gain to prevent backfill with future data
- Verify macro_df calculation


In [1]:
# 1. Enable Autoreload
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

def add_project_root_to_path():
    """Find notebooks_RLVR and add to sys.path."""
    current = Path.cwd()

    # Search upward for notebooks_RLVR folder
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR":
            sys.path.insert(0, str(path))
            print(f"✓ Added to path: {path}")
            return path
        # Also check if notebooks_RLVR exists as child (for running from stocks/)
        candidate = path / "notebooks_RLVR"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            print(f"✓ Added to path: {candidate}")
            return candidate

    raise RuntimeError("Could not find notebooks_RLVR directory")


# Run once at notebook start
add_project_root_to_path()


# 2. Force reload cached modules (run this to refresh code changes)
modules_to_reload = [
    "core.analyzer",
    "core.auditor",
    "core.contracts",    
    "core.engine",
    "core.environment",
    "core.features",
    "core.paths",
    "core.performance",
    "core.quant",
    "core.result",
    "core.settings",
    "core.utils",
    "strategy.registry",
]

for mod in modules_to_reload:
    if mod in sys.modules:
        del sys.modules[mod]


# 3. Standard imports
import pandas as pd
import numpy as np

from IPython.display import display


# 4. Fresh imports (these will re-import from disk due to cache clearing above)
from core.quant import QuantUtils
from core.analyzer import create_walk_forward_analyzer
from core.auditor import SystemAuditor as SA
from core.contracts import FilterPack, EngineInput
from core.engine import AlphaEngine, AlphaCache
from core.environment import DiscoveryEnv
from core.features import generate_features
from core.paths import OUTPUT_DIR
from core.settings import GLOBAL_SETTINGS
from core.utils import SystemUtils as SU
from strategy.registry import METRIC_REGISTRY


# 5. Pandas display settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.precision", 4)


# 6. Load data path from .env
DATA_PATH_OHLCV = SU.load_env_from_root("DATA_PATH_OHLCV")
DATA_PATH_INDICES = SU.load_env_from_root("DATA_PATH_INDICES")
print(f"DATA_PATH_OHLCV: {DATA_PATH_OHLCV}")
print(f"DATA_PATH_INDICES: {DATA_PATH_INDICES}")

# 7. Instantiate engine (customize DataFrames as needed)
master_engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)

✓ Added to path: c:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR
NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output

DATA_PATH_OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
DATA_PATH_INDICES: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet


NameError: name 'df_ohlcv' is not defined

In [3]:
df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
print(f"df_indices:|n{df_indices}")

df_indices:|n                   Adj Open  Adj High  Adj Low  Adj Close               Volume
Ticker Date                                                                   
^AXJO  1992-11-22   1455.00   1455.00  1455.00    1455.00                    0
       1992-11-23   1458.40   1458.40  1458.40    1458.40                    0
       1992-11-24   1467.90   1467.90  1467.90    1467.90                    0
       1992-11-25   1459.00   1459.00  1459.00    1459.00                    0
       1992-11-26   1458.90   1458.90  1458.90    1458.90                    0
...                     ...       ...      ...        ...                  ...
^VIX3M 2026-03-10     24.94     25.84    23.54      25.51                    0
       2026-03-11     25.29     25.98    24.93      24.97                    0
       2026-03-12     26.27     26.99    25.80      26.95                    0
       2026-03-13     25.92     27.34    25.51      27.28                    0
       2026-03-16      0.00      0.00  

In [ ]:
_indices = df_indices.index.get_level_values(0).unique().tolist()
display(_indices)
df_indices.info()

In [4]:
print(f"Takes about 1.5 minutes")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")

Takes about 1.5 minutes


In [ ]:
# print(f"df_ohlcv.head():\n {df_ohlcv.head()}\n")
df_ohlcv.info()

In [6]:
print(f"Takes about 3 minutes to generate_features")

features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
)

Takes about 3 minutes to generate_features
⚡ Generating Decoupled Features (Benchmark: SPY)...


In [ ]:
print(f"features_df.info():\n{features_df.info()}\n")
print(f"features_df.index.names:\n{features_df.index.names}\n")
print(f"macro_df.info():\n{macro_df.info()}\n")
print(f"macro_df.index.names:\n{macro_df.index.names}\n")

In [7]:
# Pre-flight Automated Audit Suite
checks = [
    SA.verify_math_integrity(),
    SA.verify_ranking_integrity(),
    SA.verify_vol_alignment_integrity(),
    SA.verify_feature_engineering_integrity(),
]

for check in checks:
    icon = "✅" if check.ok else "🔥"
    print(f"{icon} {check.msg}")
    if not check.ok:
        print("🛑 Critical Failure. System Halted.")
        break

print("=" * 85)
# Separate verify_marco_engine output from intertwine with other outputs

checks = [
    SA.verify_macro_engine(
        df_ohlcv=df_ohlcv,
        df_indices=df_indices,
        original_macro_df=macro_df,
        settings=GLOBAL_SETTINGS,
    ),
]

for check in checks:
    icon = "✅" if check.ok else "🔥"
    print(f"{icon} {check.msg}")
    if not check.ok:
        print("🛑 Critical Failure. System Halted.")
        break


--- 🛡️ Starting Feature Engineering Audit ---
⚡ Generating Decoupled Features (Benchmark: SPY)...
Audit Values:
[ nan 25.  17.5]
✅ FEATURE INTEGRITY PASSED: Wilder's ATR logic is strictly enforced.
✅ Mathematical boundaries strictly enforced.
✅ Ranking integrity strictly enforced.
✅ Reward and Risk are strictly synchronized.
✅ Wilder's ATR logic is strictly enforced.
--- Macro Verification (Benchmark: SPY) ---

Comparing verification vs original (Clip Threshold: 4.0):
✅ Mkt_Ret              | PASS (Max Diff: 0.00e+00)
✅ Macro_Trend          | PASS (Max Diff: 0.00e+00)
✅ Macro_Trend_Vel      | PASS (Max Diff: 0.00e+00)
✅ Macro_Trend_Vel_Z    | PASS (Max Diff: 0.00e+00)
✅ Macro_Trend_Mom      | PASS (Max Diff: 0.00e+00)
✅ Macro_Vix_Z          | PASS (Max Diff: 0.00e+00)
✅ Macro_Vix_Ratio      | PASS (Max Diff: 0.00e+00)
✅ Macro Integrity Verified


In [8]:
print(
    "🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)"
)

# 1. Price Matrix
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)

# 2. Volatility Matrices (Unstack and Align)
# Using features_df (the first item from the tuple)
print("   - Unstacking ATRP...")
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)

print("   - Unstacking TRP...")
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

# 3. Handle Data Gaps (Sanitize the Wide Matrices)
if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
    df_close_wide = df_close_wide.replace(0, np.nan)

# Forward fill up to the limit, then fill remaining with the "Disaster Detection" value
df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

print(
    f"✅ Pre-computation Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)
print("   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.")

🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)
   - Unstacking ATRP...
   - Unstacking TRP...
✅ Pre-computation Complete. Tickers: 1583, Days: 16158
   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.


In [9]:
# 6. Instantiate engine (customize DataFrames as needed)
master_engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)

In [10]:
# 1. Setup Parameters
decision_dt = pd.Timestamp("2025-12-10")
holding_pd = 5
lookback = 21
lookback_list = [21, 63, 252]
ticker = "SHV"  # Let's test with the ticker you verified

# A. Run via current UI logic (Manual)
# (Imagine you selected 'Sharpe (ATRP)' in the dropdown)
ui_result = master_engine.run(
    EngineInput(
        mode="Rank",
        start_date=decision_dt,
        lookback_period=lookback,
        holding_period=5,
        metric="Sharpe (ATRP)",
        benchmark_ticker="SPY",
    )
)
ui_scores = ui_result.results_df["Strategy Value"]

# B. Run via new Headless Scorer
alpha_matrix = master_engine.compute_alpha_matrix(decision_dt, lookback)
headless_scores = alpha_matrix["Sharpe (ATRP)"].reindex(ui_scores.index)

# VERIFICATION
pd.testing.assert_series_equal(ui_scores, headless_scores, check_names=False)
print("✅ Step 1 Success: Headless Scorer matches UI logic perfectly.")

DEBUG: 937 stocks passed filters on 2025-12-10
✅ Step 1 Success: Headless Scorer matches UI logic perfectly.


In [11]:
# --- TEST 2: NORMALIZATION INTEGRITY ---
raw_matrix = master_engine.compute_alpha_matrix(decision_dt, lookback)
norm_matrix = master_engine.normalize_alpha_matrix(raw_matrix)

# Verification A: Mean should be near 0
mean_check = norm_matrix.mean().abs().max()
# Verification B: Std should be near 1
std_check = norm_matrix.std().max()

print(f"Max Mean Offset: {mean_check:.6f} (Target: < 0.01)")
print(f"Max Std: {std_check:.6f} (Target: ~ 1.0)")

if mean_check < 0.01 and 0.8 < std_check < 1.2:
    print("✅ Step 2 Success: The Alpha Matrix is now 'Agent-Ready'.")
else:
    print("❌ Step 2 Failed: Normalization drift detected.")

Max Mean Offset: 0.009404 (Target: < 0.01)
Max Std: 1.000000 (Target: ~ 1.0)
✅ Step 2 Success: The Alpha Matrix is now 'Agent-Ready'.


In [12]:
# --- TEST 2.1: REGIME AWARENESS VERIFICATION ---
# decision_dt = pd.Timestamp("2025-12-10")
context = master_engine.compute_context_vector(decision_dt)

print("--- Agent Macro Context ---")
print(context)

# Check against your Plotly Image (Dec 10 values):
# Trend (Green line): Should be ~10% (0.10)
# Trend Vel (Orange line): Should be slightly below 0.0
# VIX Ratio: Chart says 0.86
# VIX Z (Purple line): Should be around -1.0

print(f"\nVerification Check:")
print(f"VIX Ratio in Context: {context['Context_Vix_Ratio'] + 1:.2f} (Target: 0.86)")

--- Agent Macro Context ---
Context_Trend        1.1401
Context_Vel_Z       -0.3619
Context_Vix_Z       -0.8380
Context_Vix_Ratio   -0.1850
dtype: float64

Verification Check:
VIX Ratio in Context: 0.81 (Target: 0.86)


In [ ]:
# # --- TEST 3 (REVISITED): VERBOSE DISCOVERY VERIFICATION ---
# # decision_dt = pd.Timestamp("2025-12-10")
# # lookback = 21

# # 1. Setup 'One-Hot' Action for Sharpe (ATRP)
# registry_keys = list(METRIC_REGISTRY.keys())
# action_weights = np.zeros(len(registry_keys))
# action_weights[registry_keys.index("Sharpe (ATRP)")] = 1.0

# # 2. Run Discovery
# discovery = master_engine.run_discovery_action(
#     decision_dt, lookback, holding_period=5, weights=action_weights
# )

# # 3. Get UI Result for verification
# ui_result = master_engine.run(
#     EngineInput(
#         mode="Ranking",
#         start_date=decision_dt,
#         lookback_period=lookback,
#         holding_period=5,
#         metric="Sharpe (ATRP)",
#         benchmark_ticker="SPY",
#     )
# )

# print(f"=== DISCOVERY ENGINE VERIFICATION (Date: {decision_dt.date()}) ===")
# print(f"Target Strategy Weights: {discovery.action_weights}")
# print(f"Selected Tickers: {discovery.selected_tickers}")
# print("-" * 50)
# print(f"Internal Discovery Score (Top 1): {discovery.metric_values.iloc[0]:.4f}")
# print(
#     f"Original UI Strategy Value (Top 1): {ui_result.results_df['Strategy Value'].iloc[0]:.4f}"
# )
# print("-" * 50)
# print(f"VERITABLE REWARD (Sharpe): {discovery.veritable_reward:.4f}")
# print(f"UI REWARD (Sharpe):        {ui_result.perf_metrics['holding_p_sharpe']:.4f}")

# if discovery.selected_tickers == ui_result.tickers:
#     print("\n✅ SELECTION MATCH: The agent and UI chose the same tickers.")

In [13]:
# --- STEP 4.1: THE VERITABLE STANDARD PROOF ---

# # 1. Setup Parameters
# decision_dt = pd.Timestamp("2025-12-10")
# holding_pd = 5
# lookback = 21
# ticker = "SHV"  # Let's test with the ticker you verified

# 2. METHOD A: The Verified UI Engine (Compounded Daily Returns)
# This is the code you already verified with Excel
ui_result = master_engine.run(
    EngineInput(
        mode="Manual List",
        start_date=decision_dt,
        lookback_period=lookback,
        holding_period=holding_pd,
        metric="Price Gain",
        manual_tickers=[ticker],
        benchmark_ticker="SPY",
    )
)
buy_date = ui_result.buy_date
end_date = ui_result.holding_end_date
ui_gain = ui_result.perf_metrics["holding_p_gain"]

# 3. METHOD B: The New Vectorized Shifter (Price Ratio)
# We calculate the log-gain directly from the price matrix
# log(P_end / P_start)
price_start = master_engine.df_close.loc[buy_date, ticker]
price_end = master_engine.df_close.loc[end_date, ticker]
vectorized_gain = np.log(price_end / price_start)

# 4. FINAL COMPARISON
print(f"=== VERITABLE STANDARD PROOF ({ticker}) ===")
print(f"Buy Date: {buy_date.date()} | End Date: {end_date.date()}")
print(f"Price at Buy: {price_start:.4f}")
print(f"Price at End: {price_end:.4f}")
print("-" * 40)
print(f"Method A (UI Engine Gain):   {ui_gain:.8f}")
print(f"Method B (Shifted Matrix):   {vectorized_gain:.8f}")
print("-" * 40)

diff = abs(ui_gain - vectorized_gain)
if diff < 1e-10:
    print("✅ VERIFICATION SUCCESS: Both methods are mathematically identical.")
    print("The Vectorized 'Time Machine' is now safe to use for RL Training.")
else:
    print(f"❌ VERIFICATION FAILED: Difference of {diff:.10f} detected.")

#

=== VERITABLE STANDARD PROOF (SHV) ===
Buy Date: 2025-12-11 | End Date: 2025-12-18
Price at Buy: 109.2710
Price at End: 109.3600
----------------------------------------
Method A (UI Engine Gain):   0.00081416
Method B (Shifted Matrix):   0.00081416
----------------------------------------
✅ VERIFICATION SUCCESS: Both methods are mathematically identical.
The Vectorized 'Time Machine' is now safe to use for RL Training.


In [14]:
# --- STEP 4.2: VECTORIZED DISCOVERY VERIFICATION ---

# A. Initialize the Time Machine (Do this once)
master_engine.precompute_reward_matrix(holding_period=5)

# decision_dt = pd.Timestamp("2025-12-10")
# lookback = 21

# B. Setup Action (Sharpe ATRP)
registry_keys = list(METRIC_REGISTRY.keys())
weights = np.zeros(len(registry_keys))
weights[registry_keys.index("Sharpe (ATRP)")] = 1.0

# C. Run Discovery (Using the NEW Vectorized functions)
# raw_matrix, discovery = master_engine.run_discovery_action(
#     decision_dt, lookback, holding_pd, weights=weights
# )


######################
# CHANGE: Only one variable on the left side
discovery = master_engine.run_discovery_action(
    decision_dt, lookback, holding_pd, weights=weights
)

# Access the matrix via the object property
raw_matrix = discovery.raw_alpha_matrix

print(f"SHV Raw Sharpe: {raw_matrix.loc['SHV', 'Sharpe (ATRP)']:.4f}")


######################


# D. Get UI Result for the "Gold Standard"
ui_result = master_engine.run(
    EngineInput(
        mode="Ranking",
        start_date=decision_dt,
        lookback_period=lookback,
        holding_period=5,
        metric="Sharpe (ATRP)",
        benchmark_ticker="SPY",
    )
)

print(f"=== VECTORIZED ENGINE VERIFICATION (Date: {decision_dt.date()}) ===")
print(
    f"Selected Tickers: {discovery.selected_tickers[:3]}... (Match UI: {discovery.selected_tickers == ui_result.tickers})"
)
print("-" * 60)

# This confirms the 0.8381 'Signal' is preserved
print(f"SIGNAL CHECK (Lookback):")
print(f"  Discovery Score (Top 1):    {discovery.metric_values.iloc[0]:.4f}")
print(
    f"  Original UI Strategy Val:   {ui_result.results_df['Strategy Value'].iloc[0]:.4f}"
)

print("-" * 60)

# This confirms the 'Reward' matches the price action
# Note: UI Reward is Sharpe (28.29), Discovery Reward is now Total Return for RL efficiency
ui_return = ui_result.perf_metrics["holding_p_gain"]

print(f"REWARD CHECK (Holding Period Return):")
print(f"  Vectorized Reward:          {discovery.veritable_reward:.8f}")
print(f"  UI Verified Gain:           {ui_return:.8f}")

if abs(discovery.veritable_reward - ui_return) < 1e-7:
    print("\n✅ VERITABLE MATCH: The Time Machine matches the UI reality.")
    print("The Engine is now ready for High-Frequency Training.")

SHV Raw Sharpe: 0.8411
DEBUG: 937 stocks passed filters on 2025-12-10
=== VECTORIZED ENGINE VERIFICATION (Date: 2025-12-10) ===
Selected Tickers: ['SHV', 'BIL', 'EXAS']... (Match UI: False)
------------------------------------------------------------
SIGNAL CHECK (Lookback):
  Discovery Score (Top 1):    4.0000
  Original UI Strategy Val:   0.8411
------------------------------------------------------------
REWARD CHECK (Holding Period Return):
  Vectorized Reward:          0.00370927
  UI Verified Gain:           0.00370927

✅ VERITABLE MATCH: The Time Machine matches the UI reality.
The Engine is now ready for High-Frequency Training.


In [15]:
# --- ENSEMBLE VERIFICATION TEST ---
# decision_dt = pd.Timestamp("2025-12-10")
# lookback_list = [21, 63, 252]

# 1. Generate the Ensemble
ensemble_vision = master_engine.compute_alpha_ensemble(decision_dt, lookback_list)

print(f"=== ENSEMBLE VISION SUMMARY (Date: {decision_dt.date()}) ===")
print(f"Total Features per Ticker: {ensemble_vision.shape[1]}")
print(f"Resolutions: {lookback_list}")
print("-" * 50)

# 2. Verify Component Integrity
# Let's check NVDA's Sharpe (ATRP) across the resolutions
nvda_stats = ensemble_vision.loc["NVDA"]

# Grab the 21d result to check against our 0.8410 benchmark
# Note: This will be the Z-scored/Clipped value, not raw 0.8410
val_21d = nvda_stats.get("21d_Sharpe (ATRP)")
val_252d = nvda_stats.get("252d_Sharpe (ATRP)")

print(f"NVDA 21d Sharpe (Z-Score):  {val_21d:.4f}")
print(f"NVDA 252d Sharpe (Z-Score): {val_252d:.4f}")

# 3. Check for Data Integrity
# Ensure we don't have all-NaN columns or duplicate data
if ensemble_vision.isna().sum().sum() == 0:
    print("\n✅ DATA INTEGRITY: Ensemble is fully populated (No NaNs).")

if val_21d != val_252d:
    print("✅ TEMPORAL DIFFERENTIATION: 21d and 252d signals are distinct.")
else:
    print("❌ ERROR: Temporal signals are identical (Slicing error?)")

# Display a sample of the 'Alpha Tensor' for one ticker
print("\nSample Feature Vector (NVDA):")
print(nvda_stats.head(10))

=== ENSEMBLE VISION SUMMARY (Date: 2025-12-10) ===
Total Features per Ticker: 33
Resolutions: [21, 63, 252]
--------------------------------------------------
NVDA 21d Sharpe (Z-Score):  -0.9867
NVDA 252d Sharpe (Z-Score): 0.1255

✅ DATA INTEGRITY: Ensemble is fully populated (No NaNs).
✅ TEMPORAL DIFFERENTIATION: 21d and 252d signals are distinct.

Sample Feature Vector (NVDA):
21d_Price Gain                      -1.1149
21d_Sharpe                          -1.1701
21d_Sharpe (ATRP)                   -0.9867
21d_Sharpe (TRP)                    -1.0317
21d_Momentum (21d)                  -1.0999
21d_Info Ratio (Stdev_Alpha, 63d)    0.2219
21d_Consistency (WinRate 5d)        -0.4973
21d_Oversold (-RSI)                  0.4814
21d_Dip Buyer (Drawdown -dd_21)      0.4004
21d_Low Volatility (-ATRP)           0.0000
Name: NVDA, dtype: float64


In [16]:
# Check if the DYNAMIC metrics differ across resolutions
print(f"NVDA 21d Price Gain (Z):  {ensemble_vision.loc['NVDA', '21d_Price Gain']:.4f}")
print(f"NVDA 252d Price Gain (Z): {ensemble_vision.loc['NVDA', '252d_Price Gain']:.4f}")

if (
    ensemble_vision.loc["NVDA", "21d_Price Gain"]
    != ensemble_vision.loc["NVDA", "252d_Price Gain"]
):
    print(
        "\n✅ SLICING VERIFIED: Dynamic window metrics are correctly calculating different time horizons."
    )

NVDA 21d Price Gain (Z):  -1.1149
NVDA 252d Price Gain (Z): 0.2630

✅ SLICING VERIFIED: Dynamic window metrics are correctly calculating different time horizons.


In [ ]:
raw_matrix.loc["SHV"]

In [23]:
# --- STEP 5: OPTIMIZED DISCOVERY WALK ---
from core.environment import DiscoveryEnv

# 1. Initialize Gym using the CACHE (The high-speed version)
# Note: We don't pass 'lookbacks' here anymore because they are inside the cache!
env = DiscoveryEnv(
    engine=master_engine, cache=cache, holding_period=5  # <--- This is the key change
)

# 2. Define the 'Discovery Loop'
obs = env.reset(start_date=pd.Timestamp("2025-01-02"))
done = False
total_steps = 0

print(f"🚀 Starting High-Speed Discovery Walk from {obs['date'].date()}...")

# Action Size: (Lookbacks * 11 Metrics) + 2 Rank Params
# Our cache has 33 features (3 lookbacks * 11 metrics)
action_size = cache.feature_cube.shape[1] + 2

while not done:
    # Agent outputting random weights
    random_action = np.random.uniform(-1, 1, size=action_size)

    # This call is now 10,000x faster!
    obs, reward, done, info = env.step(random_action)

    if total_steps % 10 == 0:
        print(
            f"Step {total_steps:03d} | Date: {info['date'].date()} | Reward: {reward:+.4f} | Tickers: {len(info.get('tickers', []))}"
        )

    total_steps += 1

print("-" * 50)
final_return = (env.equity_curve[-1] - 1) * 100
print(f"✅ Discovery Walk Complete. Total Steps: {total_steps}")
print(f"Final Agent Equity: {env.equity_curve[-1]:.4f} ({final_return:+.2f}%)")

🚀 Starting High-Speed Discovery Walk from 2025-01-02...
Step 000 | Date: 2025-01-02 | Reward: +0.0037 | Tickers: 0
Step 010 | Date: 2025-03-18 | Reward: -0.0266 | Tickers: 0
Step 020 | Date: 2025-05-29 | Reward: +0.0026 | Tickers: 0
Step 030 | Date: 2025-08-11 | Reward: +0.0476 | Tickers: 0
Step 040 | Date: 2025-10-21 | Reward: +0.0318 | Tickers: 0
Step 050 | Date: 2026-01-02 | Reward: +0.0235 | Tickers: 0
--------------------------------------------------
✅ Discovery Walk Complete. Total Steps: 59
Final Agent Equity: 2.5374 (+153.74%)


In [25]:
print(f"action_size:\n{action_size}\n")
print(f"random_action :\n{random_action }\n")
print(f"obs:\n{obs}\n")
print(f"reward:\n{reward}\n")
print(f"done:\n{done}\n")
print(f"info:\n{info}\n")
print(f"env.equity_curve:\n{env.equity_curve}\n")

action_size:
35

random_action :
[ 0.68712939  0.6987905   0.87243402  0.37378755  0.36461193 -0.27855084
 -0.68750295 -0.17589853 -0.18444633 -0.49673265  0.86354133 -0.61507948
  0.84741698 -0.32670137 -0.69495837 -0.04651186  0.35350508 -0.75166989
 -0.26890988  0.31091513 -0.11206136 -0.03825213  0.98771178  0.44849985
  0.26097708  0.12662879 -0.68876849 -0.86714789 -0.2256839   0.42873011
 -0.30291282  0.79072362 -0.37226048  0.83854906  0.570468  ]

obs:
{'ensemble':         21d_Price Gain  21d_Sharpe  21d_Sharpe (ATRP)  21d_Sharpe (TRP)  21d_Momentum (21d)  21d_Info Ratio (Stdev_Alpha, 63d)  21d_Consistency (WinRate 5d)  21d_Oversold (-RSI)  21d_Dip Buyer (Drawdown -dd_21)  21d_Low Volatility (-ATRP)  21d_VIX Filtered Momentum  63d_Price Gain  63d_Sharpe  63d_Sharpe (ATRP)  63d_Sharpe (TRP)  63d_Momentum (21d)  63d_Info Ratio (Stdev_Alpha, 63d)  63d_Consistency (WinRate 5d)  63d_Oversold (-RSI)  63d_Dip Buyer (Drawdown -dd_21)  63d_Low Volatility (-ATRP)  63d_VIX Filtered Momen

In [19]:
# 1. Initialize Cache for a specific window
cache = AlphaCache(master_engine, lookbacks=[21, 63, 252])

# We start in 2024 so the Agent has a year of "warm up" before 2025
cache.build(start_date="2024-01-01")

# 2. Initialize the Optimized Environment
env = DiscoveryEnv(engine=master_engine, cache=cache, holding_period=5)

# 3. Benchmark Step Speed
import time

obs = env.reset(start_date=pd.Timestamp("2025-01-02"))
action_size = (3 * 11) + 2
random_action = np.random.uniform(-1, 1, size=action_size)

print("\n⏱️ Starting High-Frequency Benchmark...")
start_time = time.time()
n_steps = 100

for i in range(n_steps):
    obs, reward, done, info = env.step(random_action)
    if done:
        env.reset()

end_time = time.time()
total_time = end_time - start_time

avg_step = total_time / n_steps

print(f"✅ Finished {n_steps} steps in {total_time:.4f} seconds.")
print(f"🚀 Average Step Time: {avg_step:.6f} seconds.")
print(
    f"📈 Projected Time for 1,000,000 steps: { (avg_step * 1_000_000) / 3600:.2f} hours (Previously: 9,700 hours)"
)

🏗️ Building AlphaCache for 552 days (Starting 2024-01-01)...
  Processed 0/552 days...
  Processed 20/552 days...
  Processed 40/552 days...
  Processed 60/552 days...
  Processed 80/552 days...
  Processed 100/552 days...
  Processed 120/552 days...
  Processed 140/552 days...
  Processed 160/552 days...
  Processed 180/552 days...
  Processed 200/552 days...
  Processed 220/552 days...
  Processed 240/552 days...
  Processed 260/552 days...
  Processed 280/552 days...
  Processed 300/552 days...
  Processed 320/552 days...
  Processed 340/552 days...
  Processed 360/552 days...
  Processed 380/552 days...
  Processed 400/552 days...
  Processed 420/552 days...
  Processed 440/552 days...
  Processed 460/552 days...
  Processed 480/552 days...
  Processed 500/552 days...
  Processed 520/552 days...
  Processed 540/552 days...
Timeline Error for 2026-03-13: ❌ Decision Date too late. Latest valid: 2026-03-12
Timeline Error for 2026-03-13: ❌ Decision Date too late. Latest valid: 2026-03-

In [21]:
# 1. Pick a random sample from the cache
sample_idx = 100
sample_date = cache.feature_cube.index.get_level_values("Date")[sample_idx]
sample_ticker = cache.feature_cube.index.get_level_values("Ticker")[sample_idx]

# 2. Get the value from the Cache
cached_val = cache.feature_cube.loc[(sample_date, sample_ticker)].iloc[0]

# 3. Get the value from the Engine (The slow way)
# We calculate just for this one date
engine_ensemble = master_engine.compute_alpha_ensemble(sample_date, [21, 63, 252])
engine_val = engine_ensemble.loc[sample_ticker].iloc[0]

print(f"Verification for {sample_ticker} on {sample_date.date()}:")
print(f"  Cache Value:  {cached_val:.6f}")
print(f"  Engine Value: {engine_val:.6f}")
print(f"  Difference:   {abs(cached_val - engine_val):.12f}")

assert np.isclose(cached_val, engine_val, atol=1e-8), "❌ Cache Integrity Failed!"
print("✅ Cache Integrity Verified. The fast data is identical to the trusted data.")

Verification for BK on 2024-01-02:
  Cache Value:  0.301673
  Engine Value: 0.301673
  Difference:   0.000000000000
✅ Cache Integrity Verified. The fast data is identical to the trusted data.


In [ ]:
import time

# 1. Setup
lookbacks = [21, 63, 252]
cache = AlphaCache(master_engine, lookbacks)
cache.build()  # This takes a minute, but only runs ONCE

env = DiscoveryEnv(master_engine, cache)

# 2. Benchmark the Step
obs = env.reset()
action = np.random.uniform(-1, 1, size=(len(lookbacks) * 11 + 2))

start_time = time.time()
for _ in range(100):
    _, _, done, _ = env.step(action)
    if done:
        env.reset()
end_time = time.time()

avg_step_time = (end_time - start_time) / 100
print(f"⏱️ New Average Step Time: {avg_step_time:.6f} seconds")
print(f"🚀 Speedup Factor: {35 / avg_step_time:.0f}x")

🏗️ Building AlphaCache for 16158 days...
Timeline Error for 1962-01-02: ❌ Not enough history for a 21-day lookback.
Earliest valid Decision Date: 1962-01-31
Timeline Error for 1962-01-02: ❌ Not enough history for a 63-day lookback.
Earliest valid Decision Date: 1962-04-02
Timeline Error for 1962-01-02: ❌ Not enough history for a 252-day lookback.
Earliest valid Decision Date: 1963-01-02
Timeline Error for 1962-01-03: ❌ Not enough history for a 21-day lookback.
Earliest valid Decision Date: 1962-01-31
Timeline Error for 1962-01-03: ❌ Not enough history for a 63-day lookback.
Earliest valid Decision Date: 1962-04-02
Timeline Error for 1962-01-03: ❌ Not enough history for a 252-day lookback.
Earliest valid Decision Date: 1963-01-02
Timeline Error for 1962-01-04: ❌ Not enough history for a 21-day lookback.
Earliest valid Decision Date: 1962-01-31
Timeline Error for 1962-01-04: ❌ Not enough history for a 63-day lookback.
Earliest valid Decision Date: 1962-04-02
Timeline Error for 1962-01-04

KeyboardInterrupt: 

In [ ]:
obs

In [ ]:
discovery

In [ ]:
# ui_result.perf_metrics
# registry_keys
# sharpe_atrp_idx
action_weights

In [ ]:
macro_df.loc[decision_dt]

In [ ]:
# universe_subset=None means "Scan the whole market"
analyzer1, stage1_pack = create_walk_forward_analyzer(
    master_engine, universe_subset=None
)

print("🚀 Ready for Stage 1: Run Simulation for first filter.")
analyzer1.show()

In [ ]:
# tick_price_252 = analyzer1.last_run.tickers
# tick_price_189 = analyzer1.last_run.tickers
# tick_price_126 = analyzer1.last_run.tickers
# tick_price_63 = analyzer1.last_run.tickers
# tick_price_42 = analyzer1.last_run.tickers
# tick_price_21 = analyzer1.last_run.tickers
# tick_price_10 = analyzer1.last_run.tickers

In [ ]:
# tick_sharpe_atrp_252 = analyzer1.last_run.tickers
# tick_sharpe_atrp_189 = analyzer1.last_run.tickers
# tick_sharpe_atrp_126 = analyzer1.last_run.tickers
# tick_sharpe_atrp_63 = analyzer1.last_run.tickers
# tick_sharpe_atrp_42 = analyzer1.last_run.tickers
# tick_sharpe_atrp_21 = analyzer1.last_run.tickers
# tick_sharpe_atrp_10 = analyzer1.last_run.tickers

In [ ]:
union, intersection = set_operations(
    tick_sharpe_atrp_252,
    tick_sharpe_atrp_189,
    tick_sharpe_atrp_126,
    tick_sharpe_atrp_63,
    tick_sharpe_atrp_42,
    tick_sharpe_atrp_21,
    tick_sharpe_atrp_10,
)

print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_sharpe_atrp = intersection
# int_252_189_126 = intersection

In [ ]:
print(list_to_string(int_sharpe_atrp))

In [ ]:
def list_to_string(items, separator=", ", last_separator=None):
    """
    Convert list to string with customizable separators

    Parameters:
    -----------
    items : list of strings
    separator : str, default ', '
        Separator between items
    last_separator : str, optional
        Special separator for last item (e.g., ' and ', ' or ')

    Returns:
    --------
    str : Formatted string

    Examples:
    ---------
    list_to_string(['a', 'b', 'c'])              # "a, b, c"
    list_to_string(['a', 'b', 'c'], ' | ')        # "a | b | c"
    list_to_string(['a', 'b', 'c'], ', ', ' and ')  # "a, b and c"
    """
    if not items:
        return ""

    if len(items) == 1:
        return str(items[0])

    if last_separator and len(items) > 1:
        return separator.join(items[:-1]) + last_separator + items[-1]

    return separator.join(str(item) for item in items)


# Usage examples
print(list_to_string(["a", "b", "c"]))  # a, b, c
print(list_to_string(["a", "b", "c"], " | "))  # a | b | c
print(list_to_string(["a", "b", "c"], ", ", " and "))  # a, b and c
print(list_to_string(["a", "b", "c"], ", ", ", and "))  # a, b, and c
print(list_to_string(["apple", "banana"], ", ", " and "))  # apple and banana

In [ ]:
union, intersection = set_operations(
    tick_price_252,
    tick_price_189,
    tick_price_126,
    tick_price_63,
    tick_price_42,
    tick_price_21,
    tick_price_10,
)

print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_price = intersection
# int_252_189_126 = intersection

In [ ]:
union, intersection = set_operations(
    int_sharpe_atrp,
    int_price,
)
print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_shp_atrp_price = intersection

In [ ]:
def set_operations(*lists):
    """
    Find sorted union, intersection, and elements unique to first list

    Parameters:
    -----------
    *lists : variable number of lists of strings

    Returns:
    --------
    tuple : (sorted_union, sorted_intersection, unique_to_first_list)
        - union: all unique elements from all lists
        - intersection: common elements in ALL lists
        - unique_to_first: elements only in first list, not in any other list

    Examples:
    ---------
    union, intersection, unique_first = set_operations(list1, list2)
    union, intersection, unique_first = set_operations(list1, list2, list3, list4)
    """

    if not lists:
        return [], [], []

    # Convert each list to a set
    sets = [set(lst) for lst in lists]
    first_set = sets[0]
    other_sets = sets[1:] if len(sets) > 1 else []

    # Union: all unique elements from all lists
    union_set = set().union(*sets)
    union = sorted(union_set)

    # Intersection: common elements in ALL lists
    intersection_set = first_set.intersection(*other_sets) if other_sets else first_set
    intersection = sorted(intersection_set)

    # Unique to first list: in first_set but NOT in any other set
    # Using difference: first_set - (union of all other sets)
    if other_sets:
        all_others = set().union(*other_sets)
        unique_to_first_set = (
            first_set - all_others
        )  # or first_set.difference(all_others)
    else:
        unique_to_first_set = (
            first_set  # If only one list, all elements are "unique" to it
        )

    unique_to_first = sorted(unique_to_first_set)

    return union, intersection, unique_to_first


#

In [ ]:
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# 1. LAUNCH STAGE 2 (Cascade)
# universe_subset=analyzer1.last_run.tickers means "Scan the whole market"
analyzer2, stage1_pack = create_walk_forward_analyzer(
    master_engine,
    universe_subset=analyzer1.last_run.tickers,
    # universe_subset=None,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

In [ ]:
###############################
###############################

In [ ]:
my_analyzer = analyzer1

my_res = SU.visualize_analyzer_structure(my_analyzer)

In [ ]:
def set_operations(list1, list2):
    """
    Find sorted union and intersection of two lists of strings

    Parameters:
    -----------
    list1, list2 : list of strings

    Returns:
    --------
    tuple : (sorted_union, sorted_intersection)
    """

    # Convert to sets for operations
    set1 = set(list1)
    set2 = set(list2)

    # Union: all unique elements from both lists
    union = sorted(set1 | set2)  # or set1.union(set2)

    # Intersection: common elements in both lists
    intersection = sorted(set1 & set2)  # or set1.intersection(set2)

    return union, intersection


# Example usage
list_a = ["apple", "banana", "cherry", "date", "elderberry"]
list_b = ["banana", "date", "fig", "grape", "apple"]

union, intersection = set_operations(list_a, list_b)

print(f"List 1: {list_a}")
print(f"List 2: {list_b}")
print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

In [ ]:
union, intersection = set_operations(list_a, list_b)

In [ ]:
tick_sharpe_trp_252 = SU.peek(4, my_res)

In [ ]:
analyzer1.last_run.tickers
analyzer1.last_run.start_date
analyzer1.last_run.holding_end_date

In [ ]:
# 3. Post-flight Reconciliation
audit = SA.verify_analyzer_short(my_analyzer)
if not audit.ok:
    print(f"🚨 ALERT: {audit.msg}")
    # You could trigger a notification or log the failure here

In [ ]:
audit = SA.verify_analyzer_long(my_analyzer, n_tickers=5)

In [ ]:
# Takes 4 seconds to run, checks selected tickers from analyzer1
SA.audit_feature_engineering_integrity(my_analyzer, mode="last_run")

### Audit Every Tickers in features_df, Takes 3 minutes 

In [ ]:
# # Takes 3 minutes to run, checks every tickers ~1580 tickers
# SA.audit_feature_engineering_integrity(my_analyzer, mode="system")

### Export Ticker's OHLCV and Features

In [ ]:
f_name_excel = OUTPUT_DIR / "Audit_Verification_Report.xlsx"

SU.export_audit_to_excel(audit_pack=my_analyzer.last_run, filename=f_name_excel)

In [ ]:
f_name_csv = OUTPUT_DIR / "all_tickers_data_stacked.csv"

# Single call replaces your 3 cells
file_path = SU.export_last_run_tickers_data_to_csv(
    analyzer=my_analyzer,
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    filename=f_name_csv,
)

In [ ]:
SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=my_analyzer.last_run.tickers,
    date_start=my_analyzer.last_run.start_date,
    date_end=analyzer1.last_run.holding_end_date,
    verbose=False,
)

In [ ]:
_tic = "NVDA"
_start_date = "2025-03-12"
_end_date = "2026-03-12"
print(features_df.loc[_tic][_start_date:_end_date])
# features_df.loc[_tic][_start_date:_end_date][["Ret_1d", "Consistency"]]

In [ ]:
result = SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=[_tic],
    date_start=_start_date,
    date_end=_end_date,
    verbose=False,
)
print(result[_tic])